In [ ]:
import os
import cv2
import numpy as np
import ast
import operator
import math
import time

WIDTH, HEIGHT = 1100, 700

def Webcam(cameras : int):
    if type(cameras) == int and cameras:
        return cv2.VideoCapture(cameras)
    
script_dir = os.path.dirname(os.path.abspath(os.getcwd()))
camera = Webcam(0)

if not camera.isOpened():
    raise RuntimeError("Failed to Capture Webcam")

cv2.namedWindow("Virtual Calculator", cv2.WINDOW_NORMAL)

keys = [
    ["C", "DEL", "(", ")"],
    ["7", "8", "9", "/"],
    ["4", "5", "6", "*"],
    ["1", "2", "3", "-"],
    ["0", ".", "=", "+"]
]

BUTTON_WIDTH = 90
BUTTON_HEIGHT = 72
BUTTON_GAP = 10

START_X = 635
START_Y = 215

expression = ""
result_text = ""
just_evaluated = False

last_hovered_button = None
last_activation = 0

SMOOTHING = 5

previous_x = 0
previous_y = 0

operations = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.UAdd: operator.pos,
    ast.USub: operator.neg
}


def safe_calculate(expression):

    tree = ast.parse(expression, mode="eval")

    def calculate(node):

        if isinstance(node, ast.Expression):
            return calculate(node.body)

        if (
            isinstance(node, ast.Constant)
            and type(node.value) in (int, float)
        ):
            return node.value

        if isinstance(node, ast.BinOp):

            operation = operations.get(type(node.op))

            if operation is None:
                raise ValueError("Unsupported Operation")

            return operation(
                calculate(node.left),
                calculate(node.right)
            )

        if isinstance(node, ast.UnaryOp):

            operation = operations.get(type(node.op))

            if operation is None:
                raise ValueError("Unsupported Operation")

            return operation(calculate(node.operand))

        raise ValueError("Invalid Expression")

    answer = calculate(tree)

    if not math.isfinite(float(answer)):
        raise ValueError("Invalid Result")

    if abs(answer) > 1e12:
        raise ValueError("Result is too Large")

    return f"{answer:.10g}"


def activate_button(key):

    global expression
    global result_text
    global just_evaluated

    if key == "C":

        expression = ""
        result_text = ""
        just_evaluated = False

    elif key == "DEL":

        if just_evaluated:
            result_text = ""
            just_evaluated = False

        expression = expression[:-1]

    elif key == "=":

        if expression:

            try:
                result_text = safe_calculate(expression)

            except Exception:
                result_text = "Error"

            just_evaluated = True

    else:

        if just_evaluated:

            if key in "+-*/" and result_text != "Error":
                expression = result_text
            else:
                expression = ""

            result_text = ""
            just_evaluated = False

        if len(expression) < 28:
            expression += key


while True:

    success, frame = camera.read()

    if not success:
        break

    frame = cv2.flip(frame, 1)

    frame = cv2.resize(
        frame,
        (WIDTH, HEIGHT)
    )


    hsv = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2HSV
    )

    ycrcb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2YCrCb
    )

    hsv_mask = cv2.inRange(
        hsv,
        np.array([0, 30, 60]),
        np.array([30, 200, 255])
    )

    ycrcb_mask = cv2.inRange(
        ycrcb,
        np.array([0, 125, 65]),
        np.array([255, 180, 140])
    )

    mask = cv2.bitwise_and(
        hsv_mask,
        ycrcb_mask
    )

    kernel = np.ones((5, 5), np.uint8)

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_OPEN,
        kernel
    )

    mask = cv2.morphologyEx(
        mask,
        cv2.MORPH_CLOSE,
        kernel
    )

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    fingertip = None

    if contours:

        hand = max(
            contours,
            key=cv2.contourArea
        )

        if cv2.contourArea(hand) > 3500:

            fingertip = tuple(
                hand[hand[:, :, 1].argmin()][0]
            )

            finger_x, finger_y = fingertip

            cv2.drawContours(
                frame,
                [hand],
                -1,
                (0, 255, 0),
                2
            )

            cv2.circle(
                frame,
                fingertip,
                10,
                (0, 0, 255),
                -1
            )


    button_rectangles = []

    for row_index, row in enumerate(keys):

        for col_index, key in enumerate(row):

            x1 = START_X + col_index * (
                BUTTON_WIDTH + BUTTON_GAP
            )

            y1 = START_Y + row_index * (
                BUTTON_HEIGHT + BUTTON_GAP
            )

            x2 = x1 + BUTTON_WIDTH
            y2 = y1 + BUTTON_HEIGHT

            button_rectangles.append(
                (key, x1, y1, x2, y2)
            )


    hovered_button = None

    if fingertip is not None:

        for key, x1, y1, x2, y2 in button_rectangles:

            if (
                x1 <= finger_x <= x2
                and y1 <= finger_y <= y2
            ):

                hovered_button = key
                break


    current_time = time.monotonic()

    if (
        hovered_button is not None
        and hovered_button != last_hovered_button
        and current_time - last_activation > 0.3
    ):

        activate_button(hovered_button)

        last_activation = current_time

    last_hovered_button = hovered_button


    overlay = frame.copy()

    cv2.rectangle(
        overlay,
        (610, 20),
        (1080, 665),
        (35, 35, 35),
        -1
    )

    frame = cv2.addWeighted(
        overlay,
        0.90,
        frame,
        0.10,
        0
    )

    cv2.putText(
        frame,
        "Virtual Calculator",
        (650, 60),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )


    cv2.rectangle(
        frame,
        (635, 80),
        (1025, 185),
        (245, 245, 245),
        -1
    )

    cv2.putText(
        frame,
        expression[-24:] if expression else "0",
        (650, 125),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (30, 30, 30),
        2
    )

    cv2.putText(
        frame,
        result_text,
        (650, 165),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.75,
        (0, 130, 0),
        2
    )


    for key, x1, y1, x2, y2 in button_rectangles:

        if key == hovered_button:
            button_color = (80, 190, 100)

        elif key in ["+", "-", "*", "/", "="]:
            button_color = (50, 130, 210)

        elif key in ["C", "DEL"]:
            button_color = (70, 80, 190)

        else:
            button_color = (75, 75, 75)

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            button_color,
            -1
        )

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            (255, 255, 255),
            2
        )

        text_size = cv2.getTextSize(
            key,
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            2
        )[0]

        text_x = x1 + (
            BUTTON_WIDTH - text_size[0]
        ) // 2

        text_y = y1 + (
            BUTTON_HEIGHT + text_size[1]
        ) // 2

        cv2.putText(
            frame,
            key,
            (text_x, text_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255, 255, 255),
            2
        )

    cv2.putText(
        frame,
        "Point at a button to select | Q to quit",
        (25, HEIGHT - 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        (255, 255, 255),
        2
    )

    cv2.imshow(
        "Virtual Calculator",
        frame
    )

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


camera.release()
cv2.destroyAllWindows()